# Datasets

- 380,000 Restaurants Mostly USA Based [^1]
- Uber Eats USA Restaurants [^2]
- Yelp Restaurant Dataset [^3]

[^1]: <https://www.kaggle.com/datasets/kwxdata/380k-restaurants-mostly-usa-based?select=380K_US_Restaurants.csv>
[^2]: <https://www.kaggle.com/datasets/ahmedshahriarsakib/uber-eats-usa-restaurants-menus?select=restaurants.csv>
[^3]: <https://huggingface.co/datasets/jaimik69/Yelp-Restaurant-Dataset/tree/main>

# Pre-Processing

In [20]:
# ==========================================
# 1) Imports & options (only required ones)
# ==========================================
import re, ast, json
import pandas as pd
from urllib.parse import urlparse
import usaddress
import phonenumbers  # for E.164 parsing

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

# ==========================================
# 2) Canonical schema
# ==========================================
CANON_COLS = [
    "source", "source_id",
    "name", "name_norm",
    "website", "map_url",
    "phone_raw", "phone_e164",
    "address_line1", "address_line2",
    "street", "house_number",
    "city", "state", "postal_code", "country",
    "latitude", "longitude",
    "time_zone",
    "category_primary", "categories", "price_range", "price_level",
    "rating", "rating_count",
    "is_open",
    "hours_json",
]

# ==========================================
# 3) Helper functions (only the ones used)
# ==========================================
def clean_str(x: str) -> str:
    if not isinstance(x, str):
        return ""
    x = x.strip()
    x = re.sub(r"\s+", " ", x)
    return x

def strip_company_suffixes(name: str) -> str:
    if not name:
        return name
    name = re.sub(r"\b(inc|inc\.|llc|l\.l\.c\.|corp|corp\.|co|co\.|ltd|ltd\.)\b", "", name, flags=re.I)
    return re.sub(r"\s{2,}", " ", name).strip()

def normalize_name(name: str) -> str:
    if not isinstance(name, str) or not name.strip():
        return ""
    s = name.lower()
    s = (s
         .replace("é","e").replace("è","e").replace("ê","e").replace("á","a").replace("à","a")
         .replace("ö","o").replace("ü","u").replace("ä","a").replace("ß","ss"))
    s = re.sub(r"[^\w\s]", " ", s)
    s = strip_company_suffixes(s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def normalize_url(url):
    if not isinstance(url, str) or not url.strip():
        return None
    try:
        p = urlparse(url.strip())
        if not p.scheme:
            p = urlparse("http://" + url.strip())
        return p.geturl()
    except Exception:
        return None

def to_float(x):
    try:
        return float(x)
    except Exception:
        return None

def to_int(x):
    try:
        return int(x)
    except Exception:
        return None

def categories_to_list(x):
    if x is None:
        return []
    if isinstance(x, list):
        return [clean_str(str(v)) for v in x if str(v).strip()]
    if isinstance(x, str):
        s = x.strip()
        if (s.startswith("[") and s.endswith("]")) or (s.startswith("{") and s.endswith("}")):
            try:
                obj = ast.literal_eval(s)
                if isinstance(obj, list):
                    return [clean_str(str(v)) for v in obj if str(v).strip()]
                if isinstance(obj, dict):
                    return [k for k, v in obj.items() if v in (True, "True", "true", 1, "1")]
            except Exception:
                pass
        parts = re.split(r"[|,;/]+", s)
        return [clean_str(p) for p in parts if clean_str(p)]
    return [clean_str(str(x))] if str(x).strip() else []

def parse_us_address(addr: str, fallback_city=None, fallback_state=None, fallback_zip=None):
    addr = clean_str(addr)
    if not addr:
        return (None, None, None, None, fallback_city, fallback_state, fallback_zip, "US")
    try:
        tagged, _ = usaddress.tag(addr)
        house_number = tagged.get("AddressNumber")
        street_parts = [tagged.get("StreetNamePreType"), tagged.get("StreetNamePreDirectional"),
                        tagged.get("StreetName"), tagged.get("StreetNamePostType"), tagged.get("StreetNamePostDirectional")]
        street = " ".join([p for p in street_parts if p and p.strip()]) or None
        address_line1 = " ".join([p for p in [house_number, street] if p])
        city = tagged.get("PlaceName", fallback_city)
        state = tagged.get("StateName", fallback_state)
        postal_code = tagged.get("ZipCode", fallback_zip)
        add2_parts = [tagged.get("OccupancyType"), tagged.get("OccupancyIdentifier")]
        address_line2 = " ".join([p for p in add2_parts if p and str(p) != "None"]) or None
        return (address_line1 or None, address_line2, street, house_number, city, state, postal_code, "US")
    except Exception:
        parts = [p.strip() for p in addr.split(",")]
        city = fallback_city
        state = fallback_state
        postal_code = fallback_zip
        if len(parts) >= 3:
            city = parts[-3]
            st_zip = parts[-2]
            m = re.search(r"([A-Z]{2})\s+(\d{5})(?:-\d{4})?$", st_zip)
            if m:
                state, postal_code = m.group(1), m.group(2)
        address_line1 = parts[0] if parts else None
        m2 = re.match(r"(\d+)\s+(.*)", address_line1 or "")
        house_number = m2.group(1) if m2 else None
        street = m2.group(2) if m2 else None
        return (address_line1, None, street, house_number, city, state, postal_code, "US")

def safe_literal_eval_dict(x):
    if not isinstance(x, str):
        return None
    try:
        obj = ast.literal_eval(x)
        return obj if isinstance(obj, dict) else None
    except Exception:
        return None

# --- NEW: robust phone coercion before normalization ---
def phone_coerce_string(x):
    """
    Coerce various phone cell types to a clean string before E.164:
      - floats like 13475666233.0 -> "13475666233"
      - scientific like 1.3475666233e+10 -> "13475666233"
      - strings with punctuation -> digits (keep leading '+' if present)
    Returns None if empty/NaN.
    """
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return None
    # numeric -> cast via int
    if isinstance(x, (int, float)) and not pd.isna(x):
        try:
            return str(int(x))
        except Exception:
            pass
    # everything else -> string cleanup
    s = str(x).strip()
    if not s or s.lower() in {"nan", "none", "null"}:
        return None
    if s.startswith("+"):
        digits = re.sub(r"\D", "", s)
        return "+" + digits if digits else None
    # remove all non-digits
    digits = re.sub(r"\D", "", s)
    return digits or None

def normalize_phone_e164(phone_raw: str, default_region: str = "US"):
    """Best-effort normalization of phone numbers to E.164."""
    if not isinstance(phone_raw, str) or not phone_raw.strip():
        return None
    try:
        p = phonenumbers.parse(phone_raw, default_region)
        if phonenumbers.is_possible_number(p) and phonenumbers.is_valid_number(p):
            return phonenumbers.format_number(p, phonenumbers.PhoneNumberFormat.E164)
    except Exception:
        pass
    return None

# ==========================================
# 4) Dataset-specific transformers
# ==========================================
def transform_ds1(df_raw: pd.DataFrame) -> pd.DataFrame:
    """Transform: 380k restaurants (mostly USA)."""
    df = df_raw.copy()

    rename_map = {
        "Title": "name",
        "Link": "map_url",
        "Category": "category_primary",
        "Rating": "rating",
        "Website": "website",
        "Phone": "phone_raw",
        "Address": "address_raw",
        "Images": "images",
        "Categories": "categories_raw",
        "Geo_Coordinates": "geo",
        "Time_Zone": "time_zone",
        "Latitude": "latitude",
        "Longitude": "longitude",
        "Latitude & Longitude": "latlong",
    }
    for k, v in rename_map.items():
        if k in df.columns:
            df.rename(columns={k: v}, inplace=True)

    if "geo" in df.columns:
        geo_parsed = df["geo"].apply(safe_literal_eval_dict)
        if "latitude" not in df.columns:
            df["latitude"]  = geo_parsed.apply(lambda d: d.get("latitude")  if isinstance(d, dict) else None)
        if "longitude" not in df.columns:
            df["longitude"] = geo_parsed.apply(lambda d: d.get("longitude") if isinstance(d, dict) else None)

    if "latlong" in df.columns and ("latitude" not in df or "longitude" not in df):
        ll = df["latlong"].astype(str)
        mlat = ll.str.extract(r"lat(?:itude)?[:=]\s*([\-0-9\.]+)", expand=False)
        mlon = ll.str.extract(r"lon(?:gitude)?[:=]\s*([\-0-9\.]+)", expand=False)
        if "latitude" not in df:  df["latitude"]  = mlat
        if "longitude" not in df: df["longitude"] = mlon

    if "categories_raw" in df.columns:
        df["categories_list"] = df["categories_raw"].apply(categories_to_list)
    else:
        df["categories_list"] = df.get("category_primary", "").apply(categories_to_list)
    if "category_primary" not in df.columns:
        df["category_primary"] = None
    df["category_primary"] = df["category_primary"].fillna(df["categories_list"].apply(lambda x: x[0] if x else None))
    df["categories"] = df["categories_list"]

    addr_series = df["address_raw"] if "address_raw" in df.columns else pd.Series([None]*len(df))
    addr_tuple = addr_series.apply(lambda a: parse_us_address(a))
    (df["address_line1"], df["address_line2"], df["street"], df["house_number"],
     df["city"], df["state"], df["postal_code"], df["country"]) = zip(*addr_tuple)

    df["name"] = df.get("name", "").apply(clean_str)
    df["name_norm"] = df["name"].apply(normalize_name)
    df["website"] = df.get("website").apply(normalize_url) if "website" in df.columns else None
    df["map_url"] = df.get("map_url").apply(normalize_url)

    # --- PHONE (normalize for DS1) ---
    df["phone_raw"] = df.get("phone_raw").apply(phone_coerce_string)
    df["phone_e164"] = df["phone_raw"].apply(lambda x: normalize_phone_e164(x, "US"))

    df["price_level"] = None
    df["price_range"] = None

    df["rating"] = df.get("rating").apply(to_float)
    df["rating_count"] = None

    df["latitude"]  = df.get("latitude").apply(to_float)
    df["longitude"] = df.get("longitude").apply(to_float)

    df["source"] = "kaggle_380k"
    df["source_id"] = df["map_url"].fillna(df["name"] + "|" + df["postal_code"].fillna(""))

    df["hours_json"] = None
    df["is_open"] = None
    df["time_zone"] = df.get("time_zone")

    for col in CANON_COLS:
        if col not in df.columns:
            df[col] = None
    return df[CANON_COLS]

def transform_ds2(df_raw: pd.DataFrame) -> pd.DataFrame:
    """
    Transform: Uber Eats USA Restaurants (no phone field).
    """
    df = df_raw.copy()

    rename_map = {
        "id": "source_id",
        "name": "name",
        "score": "rating",
        "ratings": "rating_count",
        "category": "categories_raw",
        "price_range": "price_range",
        "full_address": "address_raw",
        "zip_code": "postal_code",
        "lat": "latitude",
        "lng": "longitude",
    }
    for k, v in rename_map.items():
        if k in df.columns:
            df.rename(columns={k: v}, inplace=True)

    for col in ["source_id","name","rating","rating_count","categories_raw","price_range",
                "address_raw","postal_code","latitude","longitude"]:
        if col not in df.columns:
            df[col] = None

    df["categories_list"] = df.get("categories_raw").apply(categories_to_list)
    df["category_primary"] = df["categories_list"].apply(lambda x: x[0] if x else None)
    df["categories"] = df["categories_list"]

    addr_tuple = df.apply(lambda r: parse_us_address(
        r.get("address_raw"),
        fallback_city=None, fallback_state=None,
        fallback_zip=str(r.get("postal_code")) if pd.notna(r.get("postal_code")) else None
    ), axis=1)
    (df["address_line1"], df["address_line2"], df["street"], df["house_number"],
     df["city"], df["state"], df["postal_code"], df["country"]) = zip(*addr_tuple)

    df["name"] = df["name"].apply(clean_str)
    df["name_norm"] = df["name"].apply(normalize_name)
    df["website"] = None
    df["map_url"] = None

    # --- NO PHONE for DS2 ---
    df["phone_raw"] = None
    df["phone_e164"] = None

    df["price_range"] = df["price_range"]
    df["price_level"] = None

    df["rating"] = df["rating"].apply(to_float)
    df["rating_count"] = df["rating_count"].apply(to_int)

    df["latitude"]  = df["latitude"].apply(to_float)
    df["longitude"] = df["longitude"].apply(to_float)

    df["time_zone"] = None
    df["hours_json"] = None
    df["is_open"] = None

    df["source"] = "uber_eats"

    for col in CANON_COLS:
        if col not in df.columns:
            df[col] = None
    return df[CANON_COLS]

def transform_ds3_yelp_new(df_raw: pd.DataFrame) -> pd.DataFrame:
    """
    Transform: NEW Yelp CSV (yelp.csv)
    Columns: restaurant_id,name,cuisines,price,rating,review_count,phone,url,
             address,city,state,zip_code,country,latitude,longitude,source
    """
    df = df_raw.copy()

    rename_map = {
        "restaurant_id": "source_id",
        "name": "name",
        "cuisines": "categories_raw",
        "price": "price_raw",
        "rating": "rating",
        "review_count": "rating_count",
        "phone": "phone_raw",
        "url": "map_url",
        "address": "address_line1",
        "city": "city",
        "state": "state",
        "zip_code": "postal_code",
        "country": "country",
        "latitude": "latitude",
        "longitude": "longitude",
        "source": "src_in_file",
    }
    for k, v in rename_map.items():
        if k in df.columns:
            df.rename(columns={k: v}, inplace=True)

    required = ["source_id","name","categories_raw","price_raw","rating","rating_count",
                "phone_raw","map_url","address_line1","city","state","postal_code",
                "country","latitude","longitude","src_in_file"]
    for col in required:
        if col not in df.columns:
            df[col] = None

    df["categories_list"] = df.get("categories_raw").apply(categories_to_list)
    df["category_primary"] = df["categories_list"].apply(lambda x: x[0] if x else None)
    df["categories"] = df["categories_list"]

    addr_tuple = df.apply(lambda r: parse_us_address(
        r.get("address_line1"),
        fallback_city=r.get("city"),
        fallback_state=r.get("state"),
        fallback_zip=str(r.get("postal_code")) if pd.notna(r.get("postal_code")) else None
    ), axis=1)
    (df["address_line1"], df["address_line2"], df["street"], df["house_number"],
     df["city"], df["state"], df["postal_code"], df["country_from_parse"]) = zip(*addr_tuple)
    df["country"] = df["country"].fillna(df["country_from_parse"])
    df.drop(columns=["country_from_parse"], inplace=True)

    df["name"] = df["name"].apply(clean_str)
    df["name_norm"] = df["name"].apply(normalize_name)
    df["website"] = None
    df["map_url"] = df["map_url"].apply(normalize_url)

    # --- PHONE (normalize for DS3) ---
    df["phone_raw"] = df["phone_raw"].apply(phone_coerce_string)        # <-- handles floats like 134... .0
    df["phone_e164"] = df["phone_raw"].apply(lambda x: normalize_phone_e164(x, "US"))

    df["price_range"] = df["price_raw"]
    df["price_level"] = None

    df["rating"] = df["rating"].apply(to_float)
    df["rating_count"] = df["rating_count"].apply(to_int)

    df["latitude"]  = df.get("latitude").apply(to_float)
    df["longitude"] = df.get("longitude").apply(to_float)

    df["is_open"] = None
    df["hours_json"] = None
    df["time_zone"] = None

    df["source"] = df.get("src_in_file").fillna("yelp").replace("", "yelp")

    for col in CANON_COLS:
        if col not in df.columns:
            df[col] = None
    return df[CANON_COLS]

# ==========================================
# 5) Orchestration: LOCAL CSVs only
# ==========================================
def prepare_all(
    ds1_local_csv: str,
    ds2_local_csv: str,
    ds3_local_csv: str,  # new Yelp CSV
):
    """Load three local CSVs and return harmonized DataFrames for downstream integration."""
    df1_raw = pd.read_csv(ds1_local_csv)
    df2_raw = pd.read_csv(ds2_local_csv)
    df3_raw = pd.read_csv(ds3_local_csv)

    df1 = transform_ds1(df1_raw)
    df2 = transform_ds2(df2_raw)
    df3 = transform_ds3_yelp_new(df3_raw)
    return df1, df2, df3


In [21]:
df1_380k_clean, df2_uber_eats_clean, df3_yelp_clean = prepare_all(
    ds1_local_csv="../datasets/380K_US_Restaurants.csv",
    ds2_local_csv="../datasets/uber-eats-restaurants.csv",
    ds3_local_csv="../datasets/yelp_99k.csv",
)

C:\Users\Gregor Debus\AppData\Local\Temp\ipykernel_16680\464185642.py:429: DtypeWarning: Columns (3,9,10,11,12) have mixed types. Specify dtype option on import or set low_memory=False.
  df1_raw = pd.read_csv(ds1_local_csv)


In [22]:
df1_380k_clean.head(1)

,source,source_id,name,name_norm,website,map_url,phone_raw,phone_e164,address_line1,address_line2,street,house_number,city,state,postal_code,country,latitude,longitude,time_zone,category_primary,categories,price_range,price_level,rating,rating_count,is_open,hours_json
0,kaggle_380k,https://www.google.com/maps/place/Dairy+Queen+...,Dairy Queen Grill & Chill,dairy queen grill chill,http://www.fourteenfoods.com/?y_source=1_ODk5N...,https://www.google.com/maps/place/Dairy+Queen+...,+12564960404,+12564960404,3143 US-280,None,US-280,3143,Alexander City,AL,35010,US,32.93387,-85.970419,America/Chicago,Fast food restaurant,"[Fast food restaurant, Ice cream shop]",None,None,3.8,None,None,None


In [23]:
df2_uber_eats_clean.head(1)

,source,source_id,name,name_norm,website,map_url,phone_raw,phone_e164,address_line1,address_line2,street,house_number,city,state,postal_code,country,latitude,longitude,time_zone,category_primary,categories,price_range,price_level,rating,rating_count,is_open,hours_json
0,uber_eats,1,PJ Fresh (224 Daniel Payne Drive),pj fresh 224 daniel payne drive,None,None,None,None,224 Daniel Payne Drive,None,Daniel Payne Drive,224,Birmingham,AL,35207,US,33.562365,-86.830703,None,Burgers,"[Burgers, American, Sandwiches]",$,None,NaN,NaN,None,None


In [24]:
df3_yelp_clean.head(1)

,source,source_id,name,name_norm,website,map_url,phone_raw,phone_e164,address_line1,address_line2,street,house_number,city,state,postal_code,country,latitude,longitude,time_zone,category_primary,categories,price_range,price_level,rating,rating_count,is_open,hours_json
0,yelp,4MRM9BCU4x3E4e5xDRTjEQ,The Quimby,the quimby,None,https://www.yelp.com/biz/the-quimby-queens?adj...,13475666233,+13475666233,135-25 142nd St,None,142nd St,135-25,Queens,NY,11436,US,40.66735,-73.79692,None,breakfast,[breakfast],NaN,None,4.2,21,None,None


In [25]:
df1_380k_final = df1_380k_clean[['source', 'name', 'name_norm', 'website', 'map_url', 'phone_raw', 'phone_e164', 'address_line1', 'address_line2', 'street', 'house_number', 'city', 'state', 'postal_code', 'country', 'latitude', 'longitude', 'categories', 'rating']].copy()
df2_uber_eats_final = df2_uber_eats_clean[['source', 'name', 'name_norm', 'address_line1', 'address_line2', 'street', 'house_number', 'city', 'state', 'postal_code', 'country', 'latitude', 'longitude', 'categories', 'rating', 'rating_count']].copy()
df3_yelp_final = df3_yelp_clean[['source', 'name', 'name_norm', 'map_url', 'phone_raw', 'phone_e164', 'address_line1', 'address_line2', 'street', 'house_number', 'city', 'state', 'postal_code', 'country', 'latitude', 'longitude', 'categories', 'rating', 'rating_count']].copy()


# Write to parquet and add id column using PyDi Library

In [31]:
export_parquet(
    [df1_380k_final, df2_uber_eats_final, df3_yelp_final],
    ["ds1_kaggle380k_final", "ds2_ubereats_final", "ds3_yelp_final"]
)

# Export to parquet files
import os

from PyDI.io import load_parquet

# Define dataset paths
DATA_DIR = "../parquet/"

# Load Kaggle 380k dataset
kaggle380k = load_parquet(
    DATA_DIR + "ds1_kaggle380k_final.parquet",
    name="kaggle380k",
    add_index=True,
)

# Load Uber Eats dataset  
uber_eats = load_parquet(
    DATA_DIR + "ds2_ubereats_final.parquet",
    name="uber_eats",
    add_index=True,
)

# Load Yelp dataset
yelp = load_parquet(
    DATA_DIR + "ds3_yelp_final.parquet",
    name="yelp",
    add_index=True,
)

def export_parquet(dfs, names, out_dir="../parquet"):
    """Write each DataFrame to <out_dir>/<name>.parquet (one-to-one with names)."""
    if len(dfs) != len(names):
        raise ValueError(f"Length mismatch: dfs={len(dfs)} vs names={len(names)}")
    os.makedirs(out_dir, exist_ok=True)
    paths = []
    for df, n in zip(dfs, names):
        path = os.path.join(out_dir, f"{n}.parquet")
        df.to_parquet(path, index=False)
        paths.append(path)
    print(f"Wrote {len(paths)} files to {out_dir}/")
    return paths

export_parquet(
    [kaggle380k, uber_eats, yelp],
    ["kaggle380k", "uber_eats", "yelp"]
)

Wrote 3 files to ../parquet/
Wrote 3 files to ../parquet/


['../parquet\\kaggle380k.parquet',
 '../parquet\\uber_eats.parquet',
 '../parquet\\yelp.parquet']

# Test Three Way Overlap

In [9]:
import math
import pandas as pd

# ---------- Geo helpers ----------
def _haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in km; returns +inf if any coord is missing."""
    try:
        if None in (lat1, lon1, lat2, lon2):
            return float("inf")
        rlat1, rlon1, rlat2, rlon2 = map(math.radians, [lat1, lon1, lat2, lon2])
        dlat, dlon = rlat2 - rlat1, rlon2 - rlon1
        a = math.sin(dlat/2)**2 + math.cos(rlat1)*math.cos(rlat2)*math.sin(dlon/2)**2
        return 6371.0088 * (2 * math.asin(math.sqrt(a)))
    except Exception:
        return float("inf")

def _auto_decimals(tol_meters: float) -> int:
    """Choose rounding precision for candidate pruning based on tolerance."""
    if tol_meters <= 30:   return 4   # ~11 m
    if tol_meters <= 300:  return 3   # ~111 m
    if tol_meters <= 3000: return 2   # ~1.1 km
    return 1                          # ~11 km

def _prep_temp(df: pd.DataFrame, decimals: int):
    """
    Small projection (source, source_id, lat, lon, rowid, rounded lat/lon).
    Keeps the ORIGINAL row index in 'rowid' for traceability.
    """
    mask = df["latitude"].notna() & df["longitude"].notna()
    tmp = df.loc[mask, ["source", "source_id", "latitude", "longitude"]].copy()
    tmp["rowid"] = df.index[mask]
    tmp["lat_r"] = tmp["latitude"].round(decimals)
    tmp["lon_r"] = tmp["longitude"].round(decimals)
    return tmp

def _pairwise_geo_matches(dfA: pd.DataFrame, dfB: pd.DataFrame, tol_meters: float, decimals: int = None):
    """
    Find matches between dfA and dfB if distance <= tol_meters.
    Uses rounded coord join for candidates + exact Haversine filter.
    Returns (n_matches, matches_df).
    """
    tol_km = tol_meters / 1000.0
    if decimals is None:
        decimals = _auto_decimals(tol_meters)

    A = _prep_temp(dfA, decimals)
    B = _prep_temp(dfB, decimals)

    # Candidate pairs via rounded grid
    cand = A.merge(B, on=["lat_r", "lon_r"], suffixes=("_a", "_b"))
    if cand.empty:
        return 0, cand

    # Exact distance filter
    dists = cand.apply(lambda r: _haversine_km(r["latitude_a"], r["longitude_a"],
                                               r["latitude_b"], r["longitude_b"]), axis=1)
    cand = cand.loc[dists <= tol_km].copy()

    # De-duplicate pairs by original row ids
    cand = cand.drop_duplicates(subset=["rowid_a", "rowid_b"])

    # Compact output
    out = cand[[
        "source_a","source_id_a","latitude_a","longitude_a","rowid_a",
        "source_b","source_id_b","latitude_b","longitude_b","rowid_b"
    ]].reset_index(drop=True)

    return len(out), out

# ---------- Components across 3 datasets ----------
class _DSU:
    """Union-Find to build connected components from pairwise edges."""
    def __init__(self): self.p, self.r = {}, {}
    def find(self, x):
        if x not in self.p: self.p[x]=x; self.r[x]=0; return x
        while x != self.p[x]:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb: return
        if self.r[ra] < self.r[rb]: self.p[ra] = rb
        elif self.r[ra] > self.r[rb]: self.p[rb] = ra
        else: self.p[rb] = ra; self.r[ra] += 1

def _gid(source, source_id, rowid):
    """Stable node id even if source_id is missing/duplicated."""
    return f"{source}::{'' if source_id is None else str(source_id)}::{str(rowid)}"

def geo_overlap_summary_3(df1: pd.DataFrame, df2: pd.DataFrame, df3: pd.DataFrame,
                          tol_meters: float = 50.0, decimals: int = None):
    """
    Find geo-based matches across THREE harmonized DataFrames.
    Returns (summary_dict, matches_dict).
    - summary_dict has pairwise counts and number of components spanning all 3.
    - matches_dict holds the pairwise match DataFrames.
    """
    # All 3 pairs
    n12, m12 = _pairwise_geo_matches(df1, df2, tol_meters, decimals)
    n13, m13 = _pairwise_geo_matches(df1, df3, tol_meters, decimals)
    n23, m23 = _pairwise_geo_matches(df2, df3, tol_meters, decimals)

    # Build components from confirmed pairwise matches
    dsu = _DSU()
    def _union_all(md):
        if md is None or md.empty: return
        for _, r in md.iterrows():
            a = _gid(r["source_a"], r["source_id_a"], r["rowid_a"])
            b = _gid(r["source_b"], r["source_id_b"], r["rowid_b"])
            dsu.union(a, b)

    for md in (m12, m13, m23): _union_all(md)

    # Collect nodes and form components
    nodes = set()
    for md in (m12, m13, m23):
        if md is None or md.empty: continue
        for _, r in md.iterrows():
            nodes.add(_gid(r["source_a"], r["source_id_a"], r["rowid_a"]))
            nodes.add(_gid(r["source_b"], r["source_id_b"], r["rowid_b"]))

    comps = {}
    for n in nodes:
        root = dsu.find(n)
        comps.setdefault(root, set()).add(n)

    # Count how many distinct sources each component spans
    def _src_of(g): return g.split("::", 1)[0]
    span_counts = [len({_src_of(g) for g in group}) for group in comps.values()]
    three_way = sum(1 for s in span_counts if s >= 3)

    summary = {
        "tolerance_meters": tol_meters,
        "pairwise_matches": {"1_vs_2": n12, "1_vs_3": n13, "2_vs_3": n23},
        "three_way_overlap_components": three_way,
        "total_components_linked_by_geo": len(comps),
    }
    matches = {"1_vs_2": m12, "1_vs_3": m13, "2_vs_3": m23}
    return summary, matches

In [11]:
summary_geo, matches_geo = geo_overlap_summary_3(
    df1_380k_clean, df2_uber_eats_clean, df3_yelp_clean,
    tol_meters=50  # adjust if needed (e.g., 25, 100, 200)
)
print(summary_geo)

{'tolerance_meters': 50, 'pairwise_matches': {'1_vs_2': 29192, '1_vs_3': 70662, '2_vs_3': 26511}, 'three_way_overlap_components': 2915, 'total_components_linked_by_geo': 26941}


In [12]:
import re
import pandas as pd

# ---------- Address normalizers ----------
_DIR_TOKENS = {"n","s","e","w","ne","nw","se","sw","north","south","east","west"}
_SFX_MAP = {
    "street":"st","st":"st","st.":"st",
    "road":"rd","rd":"rd","rd.":"rd",
    "avenue":"ave","ave":"ave","ave.":"ave",
    "boulevard":"blvd","blvd":"blvd","blvd.":"blvd",
    "lane":"ln","ln":"ln","ln.":"ln",
    "drive":"dr","dr":"dr","dr.":"dr",
    "court":"ct","ct":"ct","ct.":"ct",
    "circle":"cir","cir":"cir","cir.":"cir",
    "place":"pl","pl":"pl","pl.":"pl",
    "parkway":"pkwy","pkwy":"pkwy","pkwy.":"pkwy","pkway":"pkwy",
    "highway":"hwy","hwy":"hwy","hwy.":"hwy",
    "terrace":"ter","ter":"ter","ter.":"ter",
    "way":"way","square":"sq","sq":"sq","sq.":"sq",
    "trail":"trl","trl":"trl","trl.":"trl",
    "alley":"aly","aly":"aly","aly.":"aly"
}

def _zip5(z):
    if z is None: return None
    m = re.search(r"(\d{5})", str(z))
    return m.group(1) if m else None

def _norm_state(s):
    return str(s).strip().upper() if s is not None and str(s).strip() else None

def _norm_city(s):
    if s is None: return None
    s = re.sub(r"[^\w\s]", " ", str(s).lower())
    s = re.sub(r"\s+", " ", s).strip()
    return s or None

def _norm_house(h):
    if h is None: return None
    s = re.sub(r"[^0-9a-z]", "", str(h).strip().lower())  # keep 123a
    return s or None

def _norm_street(street):
    if street is None: return None
    s = re.sub(r"[^\w\s]", " ", str(street).lower())
    toks = [t for t in re.split(r"\s+", s) if t]
    out = []
    for t in toks:
        if t in _DIR_TOKENS:   # drop N/S/E/W tokens
            continue
        out.append(_SFX_MAP.get(t, t))  # normalize suffixes
    s = " ".join(out).strip()
    return s or None

def _fallback_from_line1(line1):
    """If street/house are missing, try a quick split from address_line1."""
    if not isinstance(line1, str) or not line1.strip():
        return (None, None)
    m = re.match(r"\s*([0-9A-Za-z\-]+)\s+(.*)", line1.strip())
    if not m:
        return (None, _norm_street(line1))
    return (_norm_house(m.group(1)), _norm_street(m.group(2)))

def _prep_addr_temp(df: pd.DataFrame):
    """
    Build a small projection with normalized address parts.
    Does NOT modify the original dataframe.
    """
    cols = ["source","source_id","address_line1","street","house_number","city","state","postal_code"]
    tmp = df[cols].copy()

    # Fill missing street/house from address_line1 when possible; normalize both ways
    fill = tmp.apply(
        lambda r: _fallback_from_line1(r["address_line1"])
                  if (pd.isna(r["street"]) or pd.isna(r["house_number"]))
                  else (_norm_house(r["house_number"]), _norm_street(r["street"])),
        axis=1
    )
    tmp["house_norm"], tmp["street_norm"] = zip(*fill)

    tmp["city_norm"] = tmp["city"].apply(_norm_city)
    tmp["state_u"]   = tmp["state"].apply(_norm_state)
    tmp["zip5"]      = tmp["postal_code"].apply(_zip5)

    tmp["rowid"] = df.index  # preserve original index
    return tmp

# ---------- Pairwise matching by address ----------
def _pairwise_address_matches(dfA: pd.DataFrame, dfB: pd.DataFrame):
    """
    Return (n_matches, matches_df) where matches require:
      - house_norm and street_norm equal, and
      - (zip5 equal when both present) OR (city_norm + state_u equal).
    Uses exact matches on normalized fields.
    """
    A = _prep_addr_temp(dfA)
    B = _prep_addr_temp(dfB)

    # Filter rows with essential parts
    A1 = A.dropna(subset=["house_norm","street_norm"])
    B1 = B.dropna(subset=["house_norm","street_norm"])

    pairs = []

    # Path 1: ZIP available on both sides -> join on (zip5, house_norm, street_norm)
    A_zip = A1.dropna(subset=["zip5"])
    B_zip = B1.dropna(subset=["zip5"])
    p_zip = A_zip.merge(
        B_zip,
        on=["zip5","house_norm","street_norm"],
        suffixes=("_a","_b"),
        how="inner"
    )
    pairs.append(p_zip)

    # Path 2: Fallback when ZIP missing -> join on (city_norm, state_u, house_norm, street_norm)
    A_cs = A1.dropna(subset=["city_norm","state_u"])
    B_cs = B1.dropna(subset=["city_norm","state_u"])
    p_cs = A_cs.merge(
        B_cs,
        on=["city_norm","state_u","house_norm","street_norm"],
        suffixes=("_a","_b"),
        how="inner"
    )
    pairs.append(p_cs)

    cand = pd.concat(pairs, ignore_index=True) if pairs else pd.DataFrame()
    if cand.empty:
        return 0, cand

    # De-duplicate by original rows
    cand = cand.drop_duplicates(subset=["rowid_a","rowid_b"])

    # Compact view for inspection
    out = cand[[
        "source_a","source_id_a","address_line1_a","city_a","state_a","postal_code_a","rowid_a",
        "source_b","source_id_b","address_line1_b","city_b","state_b","postal_code_b","rowid_b",
    ]].reset_index(drop=True)

    return len(out), out

# ---------- Build components across 3 datasets ----------
class _DSU:
    """Union-Find to build connected components from pairwise edges."""
    def __init__(self): self.p, self.r = {}, {}
    def find(self, x):
        if x not in self.p: self.p[x]=x; self.r[x]=0; return x
        while x != self.p[x]:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb: return
        if self.r[ra] < self.r[rb]: self.p[ra] = rb
        elif self.r[ra] > self.r[rb]: self.p[rb] = ra
        else: self.p[rb] = ra; self.r[ra] += 1

def _gid(source, source_id, rowid):
    """Stable node id even if source_id is missing/duplicated."""
    return f"{source}::{'' if source_id is None else str(source_id)}::{str(rowid)}"

def address_overlap_summary_3(df1: pd.DataFrame, df2: pd.DataFrame, df3: pd.DataFrame):
    """
    Count address-based matches across THREE harmonized DataFrames:
      - Pairwise counts for (1↔2), (1↔3), (2↔3)
      - Number of connected components spanning all 3 sources
    Matching uses normalized exact rules in _pairwise_address_matches.
    """
    n12, m12 = _pairwise_address_matches(df1, df2)
    n13, m13 = _pairwise_address_matches(df1, df3)
    n23, m23 = _pairwise_address_matches(df2, df3)

    # Build components
    dsu = _DSU()
    def _union_all(md):
        if md is None or md.empty: return
        for _, r in md.iterrows():
            a = _gid(r["source_a"], r["source_id_a"], r["rowid_a"])
            b = _gid(r["source_b"], r["source_id_b"], r["rowid_b"])
            dsu.union(a, b)
    for md in (m12, m13, m23): _union_all(md)

    # Collect nodes and form components
    nodes = set()
    for md in (m12, m13, m23):
        if md is None or md.empty: continue
        for _, r in md.iterrows():
            nodes.add(_gid(r["source_a"], r["source_id_a"], r["rowid_a"]))
            nodes.add(_gid(r["source_b"], r["source_id_b"], r["rowid_b"]))

    comps = {}
    for n in nodes:
        root = dsu.find(n)
        comps.setdefault(root, set()).add(n)

    # Count components that span all three sources
    def _src_of(g): return g.split("::", 1)[0]
    span_counts = [len({_src_of(g) for g in group}) for group in comps.values()]
    three_way = sum(1 for s in span_counts if s >= 3)

    summary = {
        "pairwise_matches": {"1_vs_2": n12, "1_vs_3": n13, "2_vs_3": n23},
        "three_way_overlap_components": three_way,
        "total_components_linked_by_address": len(comps),
    }
    matches = {"1_vs_2": m12, "1_vs_3": m13, "2_vs_3": m23}
    return summary, matches

In [13]:
summary_addr, matches_addr = address_overlap_summary_3(
    df1_380k_clean, df2_uber_eats_clean, df3_yelp_clean
)
print(summary_addr)

# Inspect some 1↔3 address matches
m13 = matches_addr["1_vs_3"]
if not m13.empty:
    display(m13.head(10))

{'pairwise_matches': {'1_vs_2': 25789, '1_vs_3': 59839, '2_vs_3': 28539}, 'three_way_overlap_components': 2560, 'total_components_linked_by_address': 26666}


,source_a,source_id_a,address_line1_a,city_a,state_a,postal_code_a,rowid_a,source_b,source_id_b,address_line1_b,city_b,state_b,postal_code_b,rowid_b
0,kaggle_380k,https://www.google.com/maps/place/Louisiana+Bi...,1496 Church St,Decatur,GA,30030,896,yelp,SolJmNfUSJMz55BShAc9vg,1496 Church St,Decatur,GA,30030,22470
1,kaggle_380k,https://www.google.com/maps/place/The+Iberian+...,121 Sycamore St,Decatur,GA,30030,897,yelp,if4TMfn8v4GgOm-aKAUwrg,121 Sycamore St,Decatur,GA,30030,6370
2,kaggle_380k,https://www.google.com/maps/place/FolkArt/data...,174 W Ponce de Leon Ave,Decatur,GA,30030,898,yelp,sG4CNgTb7vu_29bcCpLRwQ,174 W Ponce De Leon Ave,Decatur,GA,30030,23479
3,kaggle_380k,https://www.google.com/maps/place/LEON%27s+Ful...,131 E Ponce de Leon Ave,Decatur,GA,30030,899,yelp,KWv9pp8Jo6O2GkfFb9OLlg,131 E Ponce De Leon Ave,Decatur,GA,30030,23464
4,kaggle_380k,https://www.google.com/maps/place/Caf%C3%A9+Li...,308 W Ponce de Leon Ave,Decatur,GA,30030,900,yelp,dphQc6E3A4s-ce1bKFSvgw,308 W Ponce De Leon Ave,Decatur,GA,30030,23467
5,kaggle_380k,https://www.google.com/maps/place/Caf%C3%A9+Li...,308 W Ponce de Leon Ave,Decatur,GA,30030,900,yelp,KmWDRT5derWIFaavkjKM1g,308 W Ponce De Leon Ave,Decatur,GA,30030,27184
6,kaggle_380k,https://www.google.com/maps/place/The+White+Bu...,123 E Court Square,Decatur,GA,30030,901,yelp,B1utosVn_70fiRUxJQRKgA,123 Court E Square,Decatur,GA,30030,6344
7,kaggle_380k,https://www.google.com/maps/place/The+Deer+and...,155 Sycamore St,Decatur,GA,30030,902,yelp,W6unkmXcb8z6iYDWzHNEMQ,155 Sycamore St,Decatur,GA,30030,6354
8,kaggle_380k,https://www.google.com/maps/place/no.+246/data...,129 E Ponce de Leon Ave,Decatur,GA,30030,903,yelp,hEH1IAmc34s3NTgQMY-iwA,129 E Ponce De Leon Ave,Decatur,GA,30030,23452
9,kaggle_380k,https://www.google.com/maps/place/Siam+Thai+Re...,123 Sycamore St,Decatur,GA,30030,904,yelp,av2wmKPV47Oh1Nw__p86MA,123 Sycamore St,Decatur,GA,30030,27138


# Find cities to scrape for

In [11]:
import re
import pandas as pd

def top_cities_across_first_two_both_present(
    df1: pd.DataFrame,
    df2: pd.DataFrame,
    df3: pd.DataFrame,
    *,
    group_by_state: bool = True,
    unify_nyc_boroughs: bool = True,
    yelp_max_present: int = 0,
    top_n: int = 200,
    return_df: bool = False
):
    """
    Prioritize cities to scrape from the Yelp API to increase THREE-WAY overlaps.

    Logic:
      1) Normalize city/state and aggregate counts per source (df1, df2, df3).
      2) Keep only cities where BOTH df1 and df2 have > 0 restaurants.
      3) Among those, select cities where df3 (current Yelp) has <= yelp_max_present (default 0).
      4) Rank by a 'priority_score' ≈ min(count_df1, count_df2) - count_df3, then total_first_two.

    Parameters
    ----------
    df1, df2, df3 : harmonized DataFrames with columns ['source','city','state'] at least.
    group_by_state : bool
        Group by (city,state) if True; otherwise by city only.
    unify_nyc_boroughs : bool
        Map {Manhattan,Brooklyn,Queens,Bronx,Staten Island} -> "New York" when state == "NY".
    yelp_max_present : int
        Max allowed existing Yelp count in df3 to consider a city as a target (0 = Yelp missing).
    top_n : int
        Return only the top-N cities by priority.
    return_df : bool
        If True, return a detailed DataFrame; otherwise return a simple list of display_city strings.

    Returns
    -------
    list[str] by default (city labels). If return_df=True, returns a DataFrame with:
      display_city, city_key, state (if grouped), total_first_two, count_df1, count_df2, count_df3,
      priority_score, and per-source columns named after the detected 'source' values.
    """
    NYC_BORO = {"manhattan","brooklyn","queens","bronx","staten island"}

    def _norm_city(s):
        if s is None: return None
        s = re.sub(r"[^\w\s]", " ", str(s).lower())
        s = re.sub(r"\s+", " ", s).strip()
        return s or None

    def _norm_state(s):
        if s is None or not str(s).strip():
            return None
        return str(s).strip().upper()

    def _prep(df):
        sub = df[["source","city","state"]].copy()
        sub["city_norm"] = sub["city"].apply(_norm_city)
        sub["state_u"]   = sub["state"].apply(_norm_state)
        if unify_nyc_boroughs:
            mask = (sub["state_u"] == "NY") & (sub["city_norm"].isin(NYC_BORO))
            sub.loc[mask, "city_norm"] = "new york"
        return sub[sub["city_norm"].notna()]

    a = _prep(df1)
    b = _prep(df2)
    c = _prep(df3)

    # Detect source labels (fallbacks if missing)
    src1 = (a["source"].dropna().unique().tolist() or ["df1"])[0]
    src2 = (b["source"].dropna().unique().tolist() or ["df2"])[0]
    src3 = (c["source"].dropna().unique().tolist() or ["df3"])[0]

    # Combine & pivot to per-source counts
    all_df = pd.concat([a, b, c], ignore_index=True)
    group_cols = ["city_norm"] + (["state_u"] if group_by_state else [])

    by_src = (
        all_df.assign(cnt=1)
              .pivot_table(index=group_cols, columns="source", values="cnt", aggfunc="sum", fill_value=0)
    )
    # Ensure 3 source columns exist
    for s in (src1, src2, src3):
        if s not in by_src.columns:
            by_src[s] = 0

    # Keep only cities present in BOTH df1 and df2
    both_idx = by_src[(by_src[src1] > 0) & (by_src[src2] > 0)].index
    by_src = by_src.loc[both_idx]

    # Filter where Yelp (df3) coverage is <= threshold
    by_src = by_src[by_src[src3] <= yelp_max_present]

    if by_src.empty:
        return [] if not return_df else by_src.reset_index().assign(
            total_first_two=0, priority_score=0, display_city=""
        )

    # Compute ranking metrics
    by_src = by_src.copy()
    by_src["total_first_two"] = by_src[src1] + by_src[src2]
    by_src["priority_score"]  = by_src[[src1, src2]].min(axis=1) - by_src[src3]
    # (priority_score >= 0 by construction; cities with bigger min(df1,df2) and smaller Yelp count rank higher)

    # Nice display label using most common original casing for city
    def _display_map(df_all):
        # build a map from (city_norm,state_u) -> nice city spelling
        tmp = (
            df_all.assign(city_disp=df_all["city"].fillna("").astype(str).str.strip())
                  .groupby(group_cols + ["city_disp"]).size()
                  .reset_index(name="n")
                  .sort_values(group_cols + ["n"], ascending=[True]*len(group_cols) + [False])
                  .groupby(group_cols).head(1)
                  .set_index(group_cols)["city_disp"]
        )
        return tmp

    disp_map = _display_map(all_df)
    out = by_src.join(disp_map.rename("display_city"))

    out = out.reset_index().rename(columns={"city_norm":"city_key", "state_u":"state"})
    # Fill display_city fallback
    out["display_city"] = out["display_city"].where(out["display_city"].astype(bool),
                                                    out["city_key"].str.title())
    if group_by_state:
        out["display_city"] = out.apply(lambda r: f"{r['display_city']}, {r['state']}", axis=1)

    # Sort by priority, then total from first two
    out = out.sort_values(["priority_score", "total_first_two"], ascending=False)

    # Limit to top_n
    if top_n is not None:
        out = out.head(top_n)

    # Rename per-source to stable columns too
    out = out.rename(columns={src1: "count_df1", src2: "count_df2", src3: "count_df3"})

    cols_order = (["display_city", "city_key"] +
                  (["state"] if group_by_state else []) +
                  ["count_df1", "count_df2", "count_df3", "total_first_two", "priority_score"])
    out = out[cols_order]

    if return_df:
        return out
    # Return just a list of city labels to scrape
    return out["display_city"].tolist()


In [17]:
import math
import pandas as pd

def _auto_decimals(tol_meters: float) -> int:
    """Choose rounding precision for grid tiles roughly matching tol_meters."""
    if tol_meters <= 30:   return 4   # ~11 m
    if tol_meters <= 300:  return 3   # ~111 m
    if tol_meters <= 3000: return 2   # ~1.1 km
    return 1                          # ~11 km

def _prep_geo_temp(df: pd.DataFrame, decimals: int):
    """
    Keep only rows with valid lat/lon and add tile keys 'lat_r', 'lon_r'.
    """
    mask = df["latitude"].notna() & df["longitude"].notna()
    tmp = df.loc[mask, ["source", "source_id", "latitude", "longitude"]].copy()
    tmp["lat_r"] = tmp["latitude"].round(decimals)
    tmp["lon_r"] = tmp["longitude"].round(decimals)
    return tmp

def prioritize_coords_for_yelp_scrape(
    df1: pd.DataFrame,
    df2: pd.DataFrame,
    df3: pd.DataFrame,
    *,
    tol_meters: float = 200.0,
    yelp_max_present: int = 0,
    top_n: int = 250,
    return_df: bool = False
):
    """
    Prioritize coordinates to query with the Yelp API to increase three-way overlaps.

    Steps
    -----
    1) Grid lat/lon into tiles by rounding (precision chosen from tol_meters).
    2) Count restaurants per tile for DF1, DF2, DF3.
    3) Keep tiles where DF1>0 AND DF2>0 AND DF3<=yelp_max_present (default: Yelp missing).
    4) Score tiles: priority = min(count_df1, count_df2) - count_df3
       (higher is better; ties broken by total_first_two = DF1+DF2).
    5) For each tile, output a representative center (mean lat/lon of DF1+DF2 points)
       and a suggested search radius (≈ tol_meters).

    Parameters
    ----------
    df1, df2, df3 : harmonized DataFrames with 'latitude' and 'longitude'
    tol_meters : float
        Target search radius you plan to use in the Yelp API; also controls tile size.
    yelp_max_present : int
        Maximum allowed existing DF3 (Yelp) count in a tile to be considered a gap.
        Use 0 to focus on tiles Yelp doesn't cover yet; raise to include under-covered tiles.
    top_n : int
        Number of highest-priority tiles to return.
    return_df : bool
        If True, return a DataFrame; else return a compact list of dicts with
        latitude, longitude, radius_m, and counts.

    Returns
    -------
    DataFrame or list[dict]:
      Columns/keys: ['lat_center','lon_center','suggested_radius_m',
                     'count_df1','count_df2','count_df3',
                     'total_first_two','priority_score','lat_r','lon_r']
    """
    decimals = _auto_decimals(tol_meters)

    A = _prep_geo_temp(df1, decimals)
    B = _prep_geo_temp(df2, decimals)
    C = _prep_geo_temp(df3, decimals)

    # Per-tile counts
    g1 = A.groupby(["lat_r", "lon_r"]).size().rename("count_df1")
    g2 = B.groupby(["lat_r", "lon_r"]).size().rename("count_df2")
    g3 = C.groupby(["lat_r", "lon_r"]).size().rename("count_df3")

    # Merge counts across tiles; fill missing with 0
    tiles = pd.concat([g1, g2, g3], axis=1).fillna(0).astype(int)

    # Keep only tiles where DF1 & DF2 both present, DF3 under threshold
    tiles = tiles[(tiles["count_df1"] > 0) & (tiles["count_df2"] > 0) & (tiles["count_df3"] <= yelp_max_present)]
    if tiles.empty:
        return tiles.reset_index() if return_df else []

    # Compute centers from DF1+DF2 points in each tile (for better query coords)
    centers_src = pd.concat([A[["lat_r","lon_r","latitude","longitude"]],
                             B[["lat_r","lon_r","latitude","longitude"]]], ignore_index=True)
    centers = centers_src.groupby(["lat_r","lon_r"])[["latitude","longitude"]].mean()
    centers = centers.rename(columns={"latitude":"lat_center","longitude":"lon_center"})

    out = tiles.join(centers, how="left")

    # Fallback center = tile corner midpoints if mean is NaN (rare)
    miss = out["lat_center"].isna() | out["lon_center"].isna()
    if miss.any():
        out.loc[miss, "lat_center"] = out.index.get_level_values("lat_r")[miss]
        out.loc[miss, "lon_center"] = out.index.get_level_values("lon_r")[miss]

    # Scores and suggested radius
    out["total_first_two"] = out["count_df1"] + out["count_df2"]
    out["priority_score"]  = out[["count_df1","count_df2"]].min(axis=1) - out["count_df3"]
    out["suggested_radius_m"] = float(tol_meters)

    # Sort by priority, then total_first_two
    out = out.sort_values(["priority_score","total_first_two"], ascending=False)

    # Trim to top_n and tidy
    if top_n is not None:
        out = out.head(top_n)
    out = out.reset_index()

    cols = ["lat_center","lon_center","suggested_radius_m",
            "count_df1","count_df2","count_df3","total_first_two","priority_score",
            "lat_r","lon_r"]
    out = out[cols]

    if return_df:
        return out

    # Compact list of targets for API calls
    targets = [
        {
            "lat": float(r.lat_center),
            "lon": float(r.lon_center),
            "radius_m": float(r.suggested_radius_m),
            "count_df1": int(r.count_df1),
            "count_df2": int(r.count_df2),
            "count_df3": int(r.count_df3),
            "priority_score": int(r.priority_score),
        }
        for r in out.itertuples(index=False)
    ]
    return targets


In [31]:
# Or get the full table with counts & priority scores
prioritized_table = top_cities_across_first_two_both_present(
    df1_380k_clean, df2_uber_eats_clean, df3_yelp_clean,
    group_by_state=True,
    unify_nyc_boroughs=True,
    yelp_max_present=0,
    top_n=100,
    return_df=True
)
prioritized_table.head(50)

,display_city,city_key,state,count_df1,count_df2,count_df3,total_first_two,priority_score
5,"Alexandria, VA",alexandria,VA,504,565,0,1069,504
77,"El Paso, TX",el paso,TX,422,763,0,1185,422
10,"Arlington, TX",arlington,TX,372,588,0,960,372
148,"Madison, WI",madison,WI,722,355,0,1077,355
81,"Everett, WA",everett,WA,372,318,0,690,318
259,"Spokane, WA",spokane,WA,246,395,0,641,246
186,"Norfolk, VA",norfolk,VA,352,246,0,598,246
288,"Virginia Beach, VA",virginia beach,VA,246,325,0,571,246
274,"Tacoma, WA",tacoma,WA,244,437,0,681,244
239,"San Antonio, TX",san antonio,TX,240,2444,0,2684,240


In [30]:
# Or inspect the full ranked table
targets_df = prioritize_coords_for_yelp_scrape(
    df1_380k_clean, df2_uber_eats_clean, df3_yelp_clean,
    tol_meters=5000, yelp_max_present=100, top_n=100, return_df=True
)
targets_df.head(50)

,lat_center,lon_center,suggested_radius_m,count_df1,count_df2,count_df3,total_first_two,priority_score,lat_r,lon_r
0,38.996575,-77.394628,5000.0,404,391,0,795,391,39.0,-77.4
1,26.205923,-98.208887,5000.0,372,317,0,689,317,26.2,-98.2
2,32.997885,-96.709906,5000.0,248,521,3,769,245,33.0,-96.7
3,32.702253,-97.111988,5000.0,242,390,0,632,242,32.7,-97.1
4,38.891371,-77.196902,5000.0,236,463,0,699,236,38.9,-77.2
5,47.681209,-117.414809,5000.0,236,283,0,519,236,47.7,-117.4
6,38.994369,-77.095476,5000.0,234,332,0,566,234,39.0,-77.1
7,38.811271,-77.085861,5000.0,220,500,0,720,220,38.8,-77.1
8,40.284941,-111.686133,5000.0,318,206,0,524,206,40.3,-111.7
9,29.529296,-95.120047,5000.0,234,254,42,488,192,29.5,-95.1


In [6]:
import os
from PyDI.io import load_parquet

# Define dataset paths
DATA_DIR = "../parquet/"

# Load Kaggle 380k dataset
kaggle380k = load_parquet(
    DATA_DIR + "kaggle380k.parquet",
    name="kaggle380k",
)

# Load Uber Eats dataset  
uber_eats = load_parquet(
    DATA_DIR + "uber_eats.parquet",
    name="uber_eats",
)

# Load Yelp dataset
yelp = load_parquet(
    DATA_DIR + "yelp.parquet",
    name="yelp",
)

In [11]:
kaggle380k.head(5)

,kaggle380k_id,source,name,name_norm,website,map_url,phone_raw,phone_e164,address_line1,address_line2,street,house_number,city,state,postal_code,country,latitude,longitude,categories,rating
0,kaggle380k-000000,kaggle_380k,Dairy Queen Grill & Chill,dairy queen grill chill,http://www.fourteenfoods.com/?y_source=1_ODk5N...,https://www.google.com/maps/place/Dairy+Queen+...,+12564960404,+12564960404,3143 US-280,None,US-280,3143,Alexander City,AL,35010,US,32.933870,-85.970419,"[Fast food restaurant, Ice cream shop]",3.8
1,kaggle380k-000001,kaggle_380k,Jake's Restaurant,jake s restaurant,http://jakesonbroad.com/,https://www.google.com/maps/place/Jake%27s+Res...,+12562344300,+12562344300,16 Broad St,None,Broad St,16,Alexander City,AL,35010,US,32.945406,-85.953806,[American restaurant],4.4
2,kaggle380k-000002,kaggle_380k,Carib Kitchen,carib kitchen,https://carib-kitchen.webnode.com/,https://www.google.com/maps/place/Carib+Kitche...,+12563924433,+12563924433,68 Broad St,None,Broad St,68,Alexander City,AL,35010,US,32.944617,-85.954932,[Caribbean restaurant],4.9
3,kaggle380k-000003,kaggle_380k,Cazadores Mexican Restaurant,cazadores mexican restaurant,None,https://www.google.com/maps/place/Cazadores+Me...,+12563923991,+12563923991,910 Cherokee Rd,None,Cherokee Rd,910,Alexander City,AL,35010,US,32.935643,-85.952365,[Mexican restaurant],4.5
4,kaggle380k-000004,kaggle_380k,La Posada Mexican Grill,la posada mexican grill,http://www.laposadamexicangrill.net/,https://www.google.com/maps/place/La+Posada+Me...,+12563293005,+12563293005,3714 US-280,None,US-280,3714,Alexander City,AL,35010,US,32.926932,-85.964919,"[Mexican restaurant, Latin American restaurant]",4.3


In [10]:
uber_eats.head(5)

,uber_eats_id,source,name,name_norm,address_line1,address_line2,street,house_number,city,state,postal_code,country,latitude,longitude,categories,rating,rating_count
0,uber_eats-00000,uber_eats,PJ Fresh (224 Daniel Payne Drive),pj fresh 224 daniel payne drive,224 Daniel Payne Drive,None,Daniel Payne Drive,224,Birmingham,AL,35207,US,33.562365,-86.830703,"[Burgers, American, Sandwiches]",NaN,NaN
1,uber_eats-00001,uber_eats,J' ti`'z Smoothie-N-Coffee Bar,j ti z smoothie n coffee bar,1521 Pinson Valley Parkway,None,Pinson Valley Parkway,1521,Birmingham,AL,35217,US,33.583640,-86.773330,"[Coffee and Tea, Breakfast and Brunch, Bubble ...",NaN,NaN
2,uber_eats-00002,uber_eats,Philly Fresh Cheesesteaks (541-B Graymont Ave),philly fresh cheesesteaks 541 b graymont ave,541-B Graymont Ave,None,Graymont Ave,541-B,Birmingham,AL,35204,US,33.509800,-86.854640,"[American, Cheesesteak, Sandwiches, Alcohol]",NaN,NaN
3,uber_eats-00003,uber_eats,Papa Murphy's (1580 Montgomery Highway),papa murphy s 1580 montgomery highway,1580 Montgomery Highway,None,Montgomery Highway,1580,Hoover,AL,35226,US,33.404439,-86.806614,[Pizza],NaN,NaN
4,uber_eats-00004,uber_eats,Nelson Brothers Cafe (17th St N),nelson brothers cafe 17th st n,314 17th St N,None,17th St N,314,Birmingham,AL,35203,US,33.514730,-86.811700,"[Breakfast and Brunch, Burgers, Sandwiches]",4.7,22.0


In [9]:
yelp.head(5)

,yelp_id,source,name,name_norm,map_url,phone_raw,phone_e164,address_line1,address_line2,street,house_number,city,state,postal_code,country,latitude,longitude,categories,rating,rating_count
0,yelp-00000,yelp,The Quimby,the quimby,https://www.yelp.com/biz/the-quimby-queens?adj...,13475666233,+13475666233,135-25 142nd St,None,142nd St,135-25,Queens,NY,11436,US,40.667350,-73.796920,[breakfast],4.2,21
1,yelp-00001,yelp,Don Peppe,don peppe,https://www.yelp.com/biz/don-peppe-south-ozone...,17188457587,+17188457587,135-58 Lefferts Blvd,None,Lefferts Blvd,135-58,South Ozone Park,NY,11420,US,40.668981,-73.821633,"[italian, seafood]",4.1,790
2,yelp-00002,yelp,Bruno Ristorante,bruno ristorante,https://www.yelp.com/biz/bruno-ristorante-howa...,17183227866,+17183227866,158-22A Crossbay Blvd,None,Crossbay Blvd,158-22A,Howard Beach,NY,11414,US,40.660154,-73.840401,[italian],4.4,378
3,yelp-00003,yelp,Nanking,nanking,https://www.yelp.com/biz/nanking-south-ozone-p...,17183233555,+17183233555,134-07 Rockaway Blvd,None,Rockaway Blvd,134-07,South Ozone Park,NY,11420,US,40.674840,-73.803980,"[chinese, thai]",3.8,411
4,yelp-00004,yelp,La Sala by Tu Casa,la sala by tu casa,https://www.yelp.com/biz/la-sala-by-tu-casa-ke...,17184873990,+17184873990,11635 Metropolitan Ave,None,Metropolitan Ave,11635,Kew Gardens,NY,11418,US,40.707369,-73.835057,[spanish],4.4,125
